In [ ]:
!pip install -q faster-whisper websockets pyngrok

In [ ]:
import asyncio
import json
import threading
import time
import re
import unicodedata

import numpy as np
import torch
import websockets

from faster_whisper import WhisperModel
from pyngrok import ngrok


PORT = 8766
SECRET = "WS_TFG_Ruben_2026_k9P4xL2mQ7vN8s"

SAMPLERATE       = 16000
BLOCK_DURATION   = 0.5
CHUNK_DURATION   = 3.0
FRAMES_PER_CHUNK = int(SAMPLERATE * CHUNK_DURATION)

SOURCE_LANGUAGE     = "es"
DEFAULT_OUTPUT_MODE = "en"   # "es" o "en"

MODEL_SIZE = "large-v3"


debug_log = []
LOG_FILE  = "/content/ws_debug.log"

def log(msg: str):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} {msg}"
    debug_log.append(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)

with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("")


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

HALLUCINATION_BLACKLIST = {
    "suscribete al canal y dale a la campanita",
    "suscribete al canal",
    "dale a la campanita",
    "gracias",
    "gracias por ver el video",
    "gracias por ver el video gracias",
    "gracias por ver el video!",
    "gracias por ver el vídeo",
    "thank you for watching",
    "thanks for watching",
    "subscribe to the channel",
}

def is_hallucination_text(text: str) -> bool:
    return normalize_text(text) in HALLUCINATION_BLACKLIST


device       = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

log(f"[Servidor] Dispositivo: {device} | compute_type: {compute_type}")

model = WhisperModel(
    MODEL_SIZE,
    device=device,
    compute_type=compute_type,
)

log(f"[Servidor] Modelo cargado correctamente: {MODEL_SIZE}")


def transcribe_and_translate(audio_data: np.ndarray, output_mode: str) -> str:
    log(f"[Servidor] Procesando chunk de {len(audio_data)} muestras | modo={output_mode}")

    # Trancripcion en espeñol
    segments_es, _ = model.transcribe(
        audio_data,
        language=SOURCE_LANGUAGE,
        beam_size=1,
        condition_on_previous_text=False,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )

    raw_texts_es = []
    last_end_es  = 0.0
    for seg in segments_es:
        raw_texts_es.append(seg.text.strip())
        last_end_es = seg.end

    transcript = " ".join(raw_texts_es).strip()
    transcript = re.sub(r"[^\w\s']", " ", transcript)
    transcript = re.sub(r"\s+", " ", transcript).strip()

    if transcript and last_end_es < (CHUNK_DURATION - 0.4):
        transcript += "."

    if transcript and is_hallucination_text(transcript):
        log(f"[Servidor] Transcripción descartada (fantasma): {transcript}")
        return ""

    if output_mode == "es":
        log(f"[Servidor] Texto final (ES): {transcript}")
        return transcript

    # Traduccion al ingles
    segments_en, _ = model.transcribe(
        audio_data,
        language=SOURCE_LANGUAGE,
        task="translate",
        beam_size=1,
        condition_on_previous_text=False,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=300),
    )

    raw_texts_en = []
    last_end_en  = 0.0
    for seg in segments_en:
        raw_texts_en.append(seg.text.strip())
        last_end_en = seg.end

    translation = " ".join(raw_texts_en).strip()
    translation = re.sub(r"[^\w\s']", " ", translation)
    translation = re.sub(r"\s+", " ", translation).strip()

    if translation and last_end_en < (CHUNK_DURATION - 0.4):
        translation += "."

    if translation and is_hallucination_text(translation):
        log(f"[Servidor] Traducción descartada (fantasma): {translation}")
        return ""

    log(f"[Servidor] Texto final (EN): {translation}")
    return translation


async def handler(websocket):
    log("[Servidor] Cliente conectado")
    buffer = []
    authenticated = False
    output_mode = DEFAULT_OUTPUT_MODE

    try:
        async for message in websocket:
            if isinstance(message, str):
                data = json.loads(message)

                if data.get("type") == "config":
                    if data.get("secret") != SECRET:
                        log("[Servidor] Auth incorrecta")
                        await websocket.send(json.dumps({
                            "type": "error", "message": "Auth incorrecta"
                        }, ensure_ascii=False))
                        await websocket.close()
                        return

                    requested_mode = data.get("output_mode")
                    if requested_mode in ("es", "en"):
                        output_mode = requested_mode

                    authenticated = True
                    log(f"[Servidor] Auth correcta | modo inicial={output_mode}")
                    await websocket.send(json.dumps({
                        "type": "ack",
                        "message": "Configuración recibida",
                        "output_mode": output_mode,
                    }, ensure_ascii=False))
                    continue

                if not authenticated:
                    await websocket.send(json.dumps({
                        "type": "error",
                        "message": "Debes autenticarte primero"
                    }, ensure_ascii=False))
                    await websocket.close()
                    return

                if data.get("type") == "set_mode":
                    requested_mode = data.get("output_mode")
                    if requested_mode in ("es", "en"):
                        output_mode = requested_mode
                        buffer = []
                        log(f"[Servidor] Modo cambiado a: {output_mode}")
                        await websocket.send(json.dumps({
                            "type": "mode_ack",
                            "output_mode": output_mode,
                        }, ensure_ascii=False))
                    continue

            else:  
                if not authenticated:
                    await websocket.send(json.dumps({
                        "type": "error",
                        "message": "Debes enviar primero la configuración"
                    }, ensure_ascii=False))
                    await websocket.close()
                    return

                block = np.frombuffer(message, dtype=np.float32)
                if block.size == 0:
                    continue

                buffer.append(block)
                total_frames = sum(len(b) for b in buffer)

                while total_frames >= FRAMES_PER_CHUNK:
                    concatenated = np.concatenate(buffer)
                    audio_chunk = concatenated[:FRAMES_PER_CHUNK].astype(np.float32)
                    remainder = concatenated[FRAMES_PER_CHUNK:]
                    buffer = [remainder] if len(remainder) > 0 else []
                    total_frames = len(remainder)

                    display_text = await asyncio.to_thread(
                        transcribe_and_translate,
                        audio_chunk,
                        output_mode,
                    )

                    if display_text:
                        await websocket.send(json.dumps({
                            "type": "result",
                            "text": display_text,
                            "output_mode": output_mode,
                        }, ensure_ascii=False))

    except websockets.ConnectionClosed:
        log("[Servidor] Cliente desconectado")
    except Exception as e:
        log(f"[Servidor] Error en handler: {e}")


async def websocket_main():
    async with websockets.serve(
        handler,
        "0.0.0.0",
        PORT,
        max_size=None,
        ping_interval=30,
        ping_timeout=30,
        compression=None,
    ):
        log(f"[Servidor] WebSocket escuchando en puerto {PORT}")
        await asyncio.Future()

def run_server():
    asyncio.run(websocket_main())


try:
    !fuser -k {PORT}/tcp
except Exception:
    pass

try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
    log("[Servidor] Túneles previos cerrados")
except Exception as e:
    log(f"[Servidor] No había túneles previos o no fue necesario cerrarlos: {e}")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)

ngrok.set_auth_token("AUTHTOKEN_AQUI")        #   Antes de continuar hay que añadir el authtoken de ngrok de la cuenta creada

public_url = ngrok.connect(PORT, bind_tls=True).public_url
ws_url = public_url.replace("https://", "wss://")

log("[Servidor] URL pública para el cliente:")
log(ws_url)
log("[Servidor] Servidor arrancado.")

Para cerrar los tuneles abiertos si es necesario:

In [ ]:
from pyngrok import ngrok

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

ngrok.kill()
print("Túneles de ngrok cerrados")